# TrOCR-style model on Kaggle (Prabhat)

Settings before you start: **Accelerator = GPU**, **Internet = On**, secret `GITHUB_TOKEN` added (see `docs/KAGGLE.md`).
Run the cells in order. Cell 6 (overfit test) must reach CER 0 before you start the full run in cell 7.
The first run downloads `microsoft/trocr-small-handwritten` from Hugging Face (about 250 MB), which is why Internet must be On.

In [ ]:
# 1. GPU check: this must print True and a GPU name
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
# 2. clone the repo (private) and install the few missing packages (Kaggle already has torch, transformers, pandas, pillow)
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
!git clone https://{token}@github.com/Nikhil04-ai/Devanagari_Word_Recognition_Project.git
%cd Devanagari_Word_Recognition_Project
!git remote set-url origin https://github.com/Nikhil04-ai/Devanagari_Word_Recognition_Project.git   # remove the token from the saved remote
!pip install -q rapidfuzz pyyaml accelerate

In [ ]:
# 3. data: IIIT-HW-Dev Version 1 (about 1.8 GB, two archives). Uses 8 parallel connections.
!python scripts/download_iiit.py --out /kaggle/working/IIIT-HW-Hindi_v1.tar.gz
!mkdir -p data/raw/iiit_hw_dev
!tar -xzf /kaggle/working/IIIT-HW-Hindi_v1.tar.gz -C data/raw/iiit_hw_dev
!tar -xzf data/raw/iiit_hw_dev/HindiSeg.tar.gz -C data/raw/iiit_hw_dev
!rm /kaggle/working/IIIT-HW-Hindi_v1.tar.gz data/raw/iiit_hw_dev/HindiSeg.tar.gz

In [ ]:
# 4. check: the CSVs point at real files (expect 69853 / 12708 / 12869 and no missing images)
import csv, os
for split in ("train", "val", "test"):
    rows = list(csv.DictReader(open(f"data/processed/words/{split}.csv", encoding="utf-8")))
    missing = sum(not os.path.exists(r["image_path"]) for r in rows[:2000])
    print(split, len(rows), "rows | missing among the first 2000:", missing)

In [ ]:
# 5. unit tests (should all pass)
!python -m pytest -q

In [ ]:
# 6. SANITY RUN: the model must memorise 100 training images (CER -> 0, it stops by itself). 1,000 updates, a few minutes on a GPU.
# If it does not reach CER 0: check pad=0 / bos=1 / eos=2 ids, that padded labels are -100, and the learning rates.
!python -m src.trocr.train --config configs/trocr/overfit.yaml

In [ ]:
# 7. FULL RUN (time budget in the config: max_minutes = 190). WATCH THE FIRST EPOCH'S TIME in the log.
# Too slow? Stop, then lower `epochs` or add a subset, for example:
#   !python -m src.trocr.train --config configs/trocr/full.yaml --train_limit 30000 --epochs 4
# (the cosine learning rate only decays fully if the run reaches its last epoch; report any subset honestly)
!python -m src.trocr.train --config configs/trocr/full.yaml
# if the session stops, run the same command again with --resume (epochs / train_limit must stay the same):
# !python -m src.trocr.train --config configs/trocr/full.yaml --resume

In [ ]:
# 8. evaluate on the TEST set ONCE; writes results/trocr_full_greedy/
!python -m src.trocr.evaluate --ckpt checkpoints/trocr_full/best.pt

In [ ]:
# 9. numbers and learning curve for report/sections/06_trocr.md (take values from here, do not invent any)
import json, pandas as pd
print(json.dumps(json.load(open("results/trocr_full_greedy/metrics.json", encoding="utf-8")), indent=1))
log = pd.read_csv("checkpoints/trocr_full/log.csv")
print(log)
ax = log.plot(x="epoch", y=["train_loss", "val_loss"], title="TrOCR loss"); ax.figure.savefig("trocr_loss.png", dpi=150, bbox_inches="tight")
ax = log.plot(x="epoch", y="val_cer", title="TrOCR validation CER"); ax.figure.savefig("trocr_val_cer.png", dpi=150, bbox_inches="tight")

In [ ]:
# 10. keep the results: copy to /kaggle/working/outputs, then Save Version -> Save & Run All, or download from the Output tab.
# Commit results/trocr_full_greedy/ (small CSV + json); never commit best.pt / last.pt.
!mkdir -p /kaggle/working/outputs
!cp -r checkpoints/trocr_full results/trocr_full_greedy trocr_loss.png trocr_val_cer.png /kaggle/working/outputs/
!ls -la /kaggle/working/outputs